# Strands Multi-Agent Workflows on AWS Bedrock — Classroom Ready Notebook

This notebook builds on the first Strands notebook and introduces **multi-agent workflows** using the **Strands Agents SDK** with **Amazon Bedrock** in SageMaker.

## Learning goals
By the end of this notebook, learners should be able to:
- understand what a multi-agent workflow is
- see the difference between a single agent and multiple specialized agents
- build an **orchestrator + specialists** pattern
- build a **sequential multi-agent pipeline**
- create two small multi-agent workflows as classroom activities


## Step 0 — Install packages

Run this only if the packages are not already installed in the SageMaker notebook kernel.

In [ ]:
# Uncomment if needed
# !pip install -q strands-agents boto3 pandas

## Step 1 — Imports

This step imports the Python packages used in the notebook.

In [1]:
import boto3
import pandas as pd
from datetime import datetime

from strands import Agent, tool
from strands.models import BedrockModel

## Step 2 — Bedrock configuration

Update the model ID if your classroom is using a different Bedrock model.

In [3]:
AWS_REGION = boto3.session.Session().region_name or "us-east-1"

'''
Avaialble Models:

amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
amazon.nova-pro-v1:0

'''

MODEL_ID = "amazon.nova-lite-v1:0"

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.1,
)

print("Region:", AWS_REGION)
print("Model ID:", MODEL_ID)

Region: us-east-1
Model ID: amazon.nova-lite-v1:0


## Step 3 — What is a multi-agent workflow?

A multi-agent workflow is a design where **multiple specialized agents** work together instead of one general-purpose agent trying to do everything.

There are two simple patterns in this notebook:

### Pattern 1 — Orchestrator + specialists
- one main agent receives the user request
- specialist agents are available as tools
- the orchestrator decides which specialist to call

### Pattern 2 — Sequential pipeline
- Agent 1 does the first task
- Agent 2 uses the output of Agent 1
- Agent 3 uses the output of Agent 2
- the final result becomes a complete workflow output

### Why use multi-agent workflows?
- each agent can stay focused
- prompts are easier to maintain
- workflows become easier to explain
- complex tasks can be split into smaller, cleaner steps

## Step 4 — Example 1: Orchestrator + specialist agents

In this example:
- a **session planner agent** creates a training session outline
- an **email writer agent** writes a communication email
- a **risk checker agent** points out delivery risks
- an **orchestrator agent** decides which specialist to use

This pattern is useful when the user may ask for different types of help in one request.

In [5]:
# Specialist 1: session planner
session_planner_agent = Agent(
    model=bedrock_model,
    name="session_planner_agent",
    description="Creates short training session outlines and agendas.",
    system_prompt="""
You are a session planning specialist.
Create short, practical training outlines.
Use simple structure:
1. Objective
2. Agenda
3. Time split
4. Output
Keep the answer concise.
""",
)

# Specialist 2: email writer
email_writer_agent = Agent(
    model=bedrock_model,
    name="email_writer_agent",
    description="Writes short professional internal emails.",
    system_prompt="""
You are an internal email writing specialist.
Write clear, short, professional emails.
Keep them classroom-friendly and easy to read.
""",
)

# Specialist 3: risk checker
risk_checker_agent = Agent(
    model=bedrock_model,
    name="risk_checker_agent",
    description="Identifies simple execution risks for workshops or training sessions.",
    system_prompt="""
You are a classroom delivery risk specialist.
List only practical risks and simple mitigations.
Keep the answer short.
""",
)

# Orchestrator
orchestrator_agent = Agent(
    model=bedrock_model,
    tools=[
        session_planner_agent,
        email_writer_agent,
        risk_checker_agent,
    ],
    system_prompt="""
You are a multi-agent orchestrator.
You have access to specialist agents:
- session_planner_agent -> use for agendas, outlines, and session plans
- email_writer_agent -> use for internal emails and formal communication
- risk_checker_agent -> use for training or workshop delivery risks

Rules:
- Use the most relevant specialist agent.
- If the user asks for more than one thing, use more than one specialist if needed.
- Combine the final answer neatly.
""",
)


In [6]:
# This will call the session_planner_agent — the orchestrator decided to call this agent on its own

orchestrator_prompt = """
I am conducting a 2-hour classroom session on prompt engineering for beginners.
Create a short session outline.
"""

orchestrator_response = orchestrator_agent(orchestrator_prompt)
print(orchestrator_response)

<thinking>The User has asked for a session outline for a 2-hour classroom session on prompt engineering for beginners. The most relevant tool for this task is the session_planner_agent.</thinking>

Tool #1: session_planner_agent
**Session Title: Introduction to Prompt Engineering for Beginners**

**Objective:**
To introduce participants to the basics of prompt engineering, enabling them to create effective prompts for various applications.

**Agenda:**
1. Introduction to Prompt Engineering
2. Key Components of a Good Prompt
3. Hands-on Activity: Crafting Effective Prompts
4. Review and Q&A

**Time Split:**
1. Introduction (15 minutes)
2. Key Components (25 minutes)
3. Hands-on Activity (45 minutes)
4. Review and Q&A (15 minutes)

**Output:**
Participants will be able to:
- Define prompt engineering.
- Identify key components of effective prompts.
- Create their own prompts for a given scenario.<thinking>The session_planner_agent has provided a detailed session outline for the 2-hour cl

In [7]:
# This prompt will call email_writer_agent - the orchestrator decided to call this agent on its own
orchestrator_prompt = """
I am conducting a 2-hour classroom session on prompt engineering for beginners.
Draft a short internal email inviting learners.
"""

orchestrator_response = orchestrator_agent(orchestrator_prompt)
print(orchestrator_response)

<thinking>The User has asked for a draft of an internal email inviting learners to a 2-hour classroom session on prompt engineering for beginners. The most relevant tool for this task is the email_writer_agent.</thinking> 
Tool #2: email_writer_agent
Subject: Invitation: 2-Hour Classroom Session on Prompt Engineering for Beginners

Dear Learners,

I hope this email finds you well.

We are excited to invite you to a 2-hour classroom session on **Prompt Engineering for Beginners**. This session is designed to introduce you to the fundamentals of prompt engineering and provide you with practical skills to enhance your learning experience.

**Date:** [Insert Date]  
**Time:** [Insert Time]  
**Location:** [Insert Classroom/Location]

During this session, you will:
- Understand the basics of prompt engineering.
- Learn techniques to create effective prompts.
- Engage in hands-on activities to practice your skills.

This is a great opportunity to enhance your skills and get started with prom

In [8]:
# This will call risk_checker_agent - the orchestrator decided to call this agent on its own
orchestrator_prompt = """
I am conducting a 2-hour classroom session on prompt engineering for beginners.
Mention 3 practical delivery risks.
"""

orchestrator_response = orchestrator_agent(orchestrator_prompt)
print(orchestrator_response)

<thinking>The User has asked for three practical delivery risks for a 2-hour classroom session on prompt engineering for beginners. The most relevant tool for this task is the risk_checker_agent.</thinking> 
Tool #3: risk_checker_agent
1. **Technical Failures**
   - **Risk:** Equipment (projector, computer, internet) may malfunction.
   - **Mitigation:** Have a backup laptop and projector; test equipment beforehand.

2. **Participant Engagement**
   - **Risk:** Participants may lose interest or struggle to follow.
   - **Mitigation:** Include interactive activities and breaks; use simple language and examples.

3. **Time Management**
   - **Risk:** Session may run over or under time.
   - **Mitigation:** Prepare a detailed agenda; use a timer to monitor progress.<thinking>The risk_checker_agent has provided three practical delivery risks for the 2-hour classroom session on prompt engineering for beginners. The User's request has been successfully completed.</thinking>

Here are three p

## Step 5 — What happened in Example 1?

In Example 1, the main orchestrator did not try to do everything itself.
Instead, it had access to specialist agents and could delegate tasks.

### What learners should notice
- each specialist has a narrow responsibility
- the orchestrator decides which specialist is appropriate
- the final answer is more modular than a single-agent answer

| agent_name | role |
| :--- | :--- |
| **session_planner_agent** | Creates training agendas and session outlines |
| **email_writer_agent** | Writes short professional internal emails |
| **risk_checker_agent** | Identifies practical classroom delivery risks |
| **orchestrator_agent** | Routes work to the correct specialist agents |

## Step 6 — Example 2: Sequential multi-agent pipeline

This example uses a pipeline with three specialized agents:

- **planner agent** -> creates a lesson plan
- **exercise generator agent** -> creates exercises from the lesson plan
- **review agent** -> reviews and improves the final material

This pattern is useful when work must happen in a fixed order.

In [10]:
lesson_planner_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a lesson planning specialist.
Create a short beginner-friendly lesson plan.
Use these sections:
1. Topic objective
2. Key concepts
3. Demo idea
4. Practice task
Keep the answer structured and short.
""",
)

exercise_generator_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are an exercise design specialist.
Based on the lesson plan provided, create:
1. two simple practice questions
2. one mini classroom activity
3. one expected learning outcome
Keep the output concise.
""",
)

review_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a teaching material reviewer.
Review the material for clarity, simplicity, and classroom usefulness.
Return:
1. strengths
2. one improvement suggestion
3. a cleaned final version
Keep the answer practical.
""",
)

In [11]:
topic = "Python functions for beginners"

# Step 1: planner creates the lesson plan
lesson_plan = lesson_planner_agent(f"Create a lesson plan for: {topic}")
print("=== LESSON PLAN ===")
print(lesson_plan)

# Step 2: exercise generator uses the lesson plan
exercise_pack = exercise_generator_agent(
    f"Use this lesson plan and create exercises:\n\n{lesson_plan}"
)
print("\n=== EXERCISE PACK ===")
print(exercise_pack)

# Step 3: reviewer reviews the full package
reviewed_output = review_agent(
    f"Review this lesson plan and exercise pack for classroom delivery:\n\nLesson Plan:\n{lesson_plan}\n\nExercises:\n{exercise_pack}"
)
print("\n=== REVIEWED OUTPUT ===")
print(reviewed_output)

**Lesson Plan: Introduction to Python Functions**

**1. Topic Objective:**
By the end of this lesson, students will be able to understand the concept of functions in Python, create simple functions, and call them with different arguments.

**2. Key Concepts:**
- Definition of a function
- Function syntax (`def`)
- Parameters and arguments
- Return values
- Calling a function
- Scope of variables

**3. Demo Idea:**
- **Introduction (5 minutes):** Briefly explain what functions are and why they are useful.
- **Live Coding (15 minutes):** 
  - Create a simple function to add two numbers.
  - Show how to call the function with different arguments.
  - Modify the function to return a value and print the result.
  
  ```python
  def add_numbers(a, b):
      return a + b

  result = add_numbers(3, 5)
  print(result)  # Output: 8
  ```

**4. Practice Task:**
- **Task (15 minutes):** 
  - Ask students to create a function that takes two numbers and returns their product.
  - Then, ask them to c

## Step 6 — What happened in Example 2?

Example 2 is different from the orchestrator pattern.

Here, work follows a **fixed sequence**:
1. planning
2. exercise generation
3. review

### What learners should notice
- each agent receives the previous output as input
- this is useful when the task has a natural order
- the pipeline is easier to control because the sequence is explicit

| step | agent | purpose |
| :--- | :--- | :--- |
| 1 | **lesson_planner_agent** | Create the first structured lesson plan |
| 2 | **exercise_generator_agent** | Generate exercises from the lesson plan |
| 3 | **review_agent** | Review and improve the final material |

## Step 7 — Compare the two multi-agent patterns

This table helps learners compare:
- orchestrator + specialists
- sequential pipeline

| pattern | how_it_works | best_for | control_style |
| :--- | :--- | :--- | :--- |
| **Orchestrator + specialists** | One main agent decides which specialist agent to call | Mixed user requests where routing is needed | Flexible |
| **Sequential pipeline** | Agents run in a fixed order and pass outputs forward | Tasks with clear step-by-step flow | Structured |

## Step 8 — Activity 1: Build a helpdesk triage multi-agent workflow

### Goal
Build an orchestrator with two specialist agents:
- one agent for **technical troubleshooting**
- one agent for **email response drafting**

### What learners should do
1. complete the two specialist agents
2. create the orchestrator
3. test it with two prompts

### Suggested prompts
- “My IDE is not opening. Help me troubleshoot.”
- “Write a short internal email telling the team that the server will restart at 6 PM.”

In [ ]:
# Your code goes here

## Step 9 - Activity 2 : A consulting company frequently responds to RFPs from clients but struggles with:

You need to build a multi-agent system that automates the end-to-end RFP response creation process, with built-in review and feedback loops.

RFP Parser Tool
Extract requirements from raw input

Outline Generator Tool
Convert requirements → structured plan

Draft Generator Tool
Create sections of proposal

Evaluation Tool
Return score + structured feedback

Refinement Tool
Improve draft using feedback

In [ ]:
# Your code goes here